In [5]:
import pandas as pd
import pyodbc

from sqlalchemy import create_engine
from urllib.parse import quote_plus


server = r"Nimi\SQLEXPRESS"
database = "DataWarehouse"


connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)


params = quote_plus(connection_string)


engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}"
)

In [7]:
fact_sales = pd.read_sql(
    "SELECT * FROM gold.fact_sales",
    engine
)

dim_customers = pd.read_sql(
    "SELECT * FROM gold.dim_customers",
    engine
)

dim_products = pd.read_sql(
    "SELECT * FROM gold.dim_products",
    engine
)

dim_date = pd.read_sql(
    "SELECT * FROM gold.dim_date",
    engine
)

In [9]:
dim_customers.info()
dim_products.info()
dim_date.info()

<class 'pandas.DataFrame'>
RangeIndex: 18484 entries, 0 to 18483
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   customer_key     18484 non-null  int64 
 1   customer_id      18484 non-null  int64 
 2   customer_number  18484 non-null  str   
 3   first_name       18484 non-null  str   
 4   last_name        18484 non-null  str   
 5   country          18484 non-null  str   
 6   marital_status   18484 non-null  str   
 7   gender           18484 non-null  str   
 8   birthdate        18467 non-null  object
 9   create_date      18484 non-null  object
dtypes: int64(2), object(2), str(6)
memory usage: 1.4+ MB
<class 'pandas.DataFrame'>
RangeIndex: 295 entries, 0 to 294
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   product_key     295 non-null    int64  
 1   product_id      295 non-null    int64  
 2   product_number  295 non-n

In [11]:
dim_customers.isna().sum()

customer_key        0
customer_id         0
customer_number     0
first_name          0
last_name           0
country             0
marital_status      0
gender              0
birthdate          17
create_date         0
dtype: int64

In [8]:
print("Fact sales:", fact_sales.shape)
print("Customers:", dim_customers.shape)
print("Products:", dim_products.shape)
print("Dates:", dim_date.shape)

Fact sales: (60398, 12)
Customers: (18484, 10)
Products: (295, 11)
Dates: (1139, 10)


In [12]:
null_summary = pd.DataFrame({
    "null_count": fact_sales.isna().sum(),
    "null_percent": (
        fact_sales.isna().mean() * 100
    ).round(2)
})

null_summary

,null_count,null_percent
order_number,0,0.00
product_key,0,0.00
customer_key,0,0.00
order_date_key,19,0.03
shipping_date_key,0,0.00
due_date_key,0,0.00
order_date,19,0.03
shipping_date,0,0.00
due_date,0,0.00
sales_amount,0,0.00


In [14]:
fact_sales.duplicated().sum()

np.int64(0)

In [15]:
len(fact_sales)

60398

In [16]:
fact_sales["order_number"].nunique()

27659

In [17]:
fact_sales["customer_key"].nunique()
fact_sales["product_key"].nunique()

130

In [19]:
total_revenue = fact_sales["sales_amount"].sum()

total_orders = fact_sales["order_number"].nunique()

total_quantity = fact_sales["quantity"].sum()

active_customers = fact_sales["customer_key"].nunique()

products_sold = fact_sales["product_key"].nunique()

average_order_value = total_revenue / total_orders

In [20]:
kpi_summary = pd.DataFrame({
    "Metric": [
        "Total Revenue",
        "Total Orders",
        "Total Quantity",
        "Active Customers",
        "Products Sold",
        "Average Order Value"
    ],
    "Value": [
        total_revenue,
        total_orders,
        total_quantity,
        active_customers,
        products_sold,
        average_order_value
    ]
})

kpi_summary

,Metric,Value
0,Total Revenue,2.935625e+07
1,Total Orders,2.765900e+04
2,Total Quantity,6.042300e+04
3,Active Customers,1.848400e+04
4,Products Sold,1.300000e+02
5,Average Order Value,1.061363e+03


In [21]:
fact_sales.duplicated(
    subset=["order_number", "product_key"]
).sum()

np.int64(0)

In [22]:
(
    fact_sales
    .groupby(["order_number", "product_key"])
    .size()
    .reset_index(name="row_count")
    .query("row_count > 1")
    .sort_values("row_count", ascending=False)
    .head(20)
)

,order_number,product_key,row_count


In [23]:
lines_per_order = (
    fact_sales
    .groupby("order_number")
    .size()
)

lines_per_order.head()

order_number
SO43697    1
SO43698    1
SO43699    1
SO43700    1
SO43701    1
dtype: int64

In [24]:
total_orders = fact_sales["order_number"].nunique()
total_lines = len(fact_sales)
total_units = fact_sales["quantity"].sum()

print("Orders:", total_orders)
print("Sales lines:", total_lines)
print("Units sold:", total_units)

Orders: 27659
Sales lines: 60398
Units sold: 60423


In [25]:
avg_lines_per_order = (
    len(fact_sales)
    / fact_sales["order_number"].nunique()
)

round(avg_lines_per_order, 2)

2.18

In [26]:
fact_sales.duplicated().sum()

np.int64(0)

In [27]:
fact_sales.duplicated(
    subset=["order_number", "product_key"]
).sum()

np.int64(0)

In [28]:
fact_sales["quantity"].sum()

np.int64(60423)

In [29]:
total_revenue = fact_sales["sales_amount"].sum()

total_orders = fact_sales["order_number"].nunique()

total_units = fact_sales["quantity"].sum()

active_customers = fact_sales["customer_key"].nunique()

products_sold = fact_sales["product_key"].nunique()

average_order_value = total_revenue / total_orders

average_units_per_order = total_units / total_orders

In [30]:
print(f"Total Revenue: {total_revenue:,.2f}")
print(f"Total Orders: {total_orders:,}")
print(f"Units Sold: {total_units:,}")
print(f"Active Customers: {active_customers:,}")
print(f"Products Sold: {products_sold:,}")
print(f"Average Order Value: {average_order_value:,.2f}")
print(f"Average Units per Order: {average_units_per_order:.2f}")

Total Revenue: 29,356,250.00
Total Orders: 27,659
Units Sold: 60,423
Active Customers: 18,484
Products Sold: 130
Average Order Value: 1,061.36
Average Units per Order: 2.18


In [33]:
active_customers = fact_sales["customer_key"].nunique()
print(active_customers)

18484


In [32]:
len(dim_customers)

18484

In [34]:
total_customers = len(dim_customers)

active_customers = fact_sales[
    "customer_key"
].nunique()

customers_without_sales = (
    total_customers - active_customers
)

print(f"Total customers: {total_customers:,}")
print(f"Customers with sales: {active_customers:,}")
print(f"Customers without sales: {customers_without_sales:,}")

Total customers: 18,484
Customers with sales: 18,484
Customers without sales: 0


In [36]:
customer_revenue = (
    fact_sales.groupby("customer_key")["sales_amount"].sum()
)
customer_revenue.head()

customer_key
1    8249.0
2    6384.0
3    8114.0
4    8139.0
5    8196.0
Name: sales_amount, dtype: float64

In [40]:
fact_sales["order_date"].dtype

dtype('<M8[s]')

In [38]:
fact_sales["order_date"] = pd.to_datetime(
    fact_sales["order_date"]
)

In [41]:
fact_sales["order_year"] = (
    fact_sales["order_date"].dt.year
)

fact_sales["order_month"] = (
    fact_sales["order_date"].dt.month
)

In [42]:
yearly_sales = (
    fact_sales
    .groupby("order_year")
    .agg(
        revenue=("sales_amount", "sum"),
        orders=("order_number", "nunique"),
        units=("quantity", "sum"),
        customers=("customer_key", "nunique")
    )
    .reset_index()
)

yearly_sales

,order_year,revenue,orders,units,customers
0,2010.0,43419.0,14,14,14
1,2011.0,7075088.0,2216,2216,2216
2,2012.0,5842231.0,3269,3397,3255
3,2013.0,16344878.0,21287,52807,17427
4,2014.0,45642.0,871,1970,834
